# Práctica Final — Análisis de Sentimientos con LLMs
## Modelo 2: `microsoft/Phi-3-mini-4k-instruct`

---

**Asignatura:** Búsqueda y Análisis de Información  
**Autores:** Juan Battaglio Quintana, Juan Guasp Timoner, Alejandro Parra Purroy  
**Fecha:** Mayo 2026  

---

### Descripción del modelo

**Phi-3-mini-4k-instruct** es un modelo de lenguaje de Microsoft lanzado en **abril de 2024**, con **3.8 mil millones de parámetros** y una ventana de contexto de **4096 tokens**. Pertenece a la familia Phi-3, diseñada para maximizar el rendimiento en tareas de razonamiento con un tamaño reducido. La variante `instruct` ha sido ajustada mediante RLHF para seguir instrucciones en formato conversacional.

| Característica | Valor |
|---|---|
| Parámetros | 3.8B |
| Fecha de lanzamiento | Abril 2024 |
| Ventana de contexto | 4096 tokens |
| Tipo | Decoder-only, instruction-tuned |
| Arquitectura | Transformer con grouped-query attention |
| Licencia | MIT |

### Tarea abordada

Clasificación de mensajes en español según tres dimensiones:
- **Sentimiento**: tristeza / alegría / neutralidad
- **Subjetividad**: objetivo / subjetivo
- **Polaridad**: positivo / negativo / neutro

### Estrategias de prompt engineering evaluadas

1. **Prompt base**: instrucción mínima, sin estructura definida
2. **Prompt plantilla**: estructura formal con Tarea, Contexto, Restricciones, Formato y Criterio de calidad
3. **Prompt de razonamiento**: análisis paso a paso con contraste de alternativas y conclusión justificada

### Índice

0. Instalación de dependencias
1. Configuración del entorno
2. Carga del modelo
3. Dataset y etiquetas de referencia
4. Definición de prompts
5. Parámetros y función de inferencia
6. Ejecución del experimento
7. Métricas objetivas
8. Análisis de variabilidad
9. Visualizaciones
10. Conclusiones

---
## 0. Instalacion de dependencias

Ejecuta esta celda si alguna de las librerías necesarias no está instalada en tu entorno.
En **Google Colab** se recomienda ejecutarla siempre. En **entorno local** solo es necesaria la primera vez.

> **Nota:** Después de la instalación puede ser necesario **reiniciar el kernel** antes de continuar.

In [2]:
# Instalar versiones específicas compatibles entre sí para Phi-3-mini
# transformers 4.44.x + huggingface_hub < 0.30 evitan el StrictDataclassDefinitionError

!pip install \
    "transformers==4.44.2" \
    "huggingface_hub==0.24.6" \
    "accelerate>=0.26.0" \
    sentencepiece protobuf -q

# Verificar versiones instaladas
import importlib
for pkg in ['torch', 'transformers', 'huggingface_hub', 'accelerate', 'sentencepiece']:
    try:
        mod = importlib.import_module(pkg)
        print(f'  {pkg}: {getattr(mod, "__version__", "instalado")}')
    except ImportError:
        print(f'  {pkg}: NO instalado')

print("\nREINICIA el runtime de Colab antes de continuar (Entorno de ejecución → Reiniciar sesión)")

  torch: 2.10.0+cu128
  transformers: 4.44.2
  huggingface_hub: 0.24.6
  accelerate: 1.13.0
  sentencepiece: 0.2.1

REINICIA el runtime de Colab antes de continuar (Entorno de ejecución → Reiniciar sesión)


---
## 1. Configuración del entorno <a id='sec1'></a>

Importamos las librerías necesarias y detectamos el hardware disponible.

In [3]:
import torch
import time
import re
import unicodedata
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline

# Detectar dispositivo
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Dispositivo: {device}")

if device == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    vram_total = torch.cuda.get_device_properties(0).total_memory / 1e9
    vram_libre = torch.cuda.mem_get_info()[0] / 1e9
    print(f"VRAM total: {vram_total:.1f} GB")
    print(f"VRAM libre: {vram_libre:.1f} GB")
else:
    print("ADVERTENCIA: No se detectó GPU. El modelo tardará considerablemente más en CPU.")

print(f"\nPyTorch version: {torch.__version__}")

Dispositivo: cuda
GPU: Tesla T4
VRAM total: 15.6 GB
VRAM libre: 15.5 GB

PyTorch version: 2.10.0+cu128


---
## 2. Carga del modelo <a id='sec2'></a>

Cargamos `microsoft/Phi-3-mini-4k-instruct` desde HuggingFace. El modelo ocupa aproximadamente **7.6 GB en float16** en VRAM. Si tu GPU tiene menos de 8 GB, considera usar `torch.float32` en CPU o añadir cuantización con `bitsandbytes`.

> **Nota:** La primera ejecución descargará los pesos del modelo (~7 GB). Las ejecuciones posteriores los cargarán desde caché local.

In [4]:
MODEL_ID = "microsoft/Phi-3-mini-4k-instruct"

print(f"Cargando tokenizador de '{MODEL_ID}'...")
# trust_remote_code=False: usar implementación nativa de transformers >= 4.40
# Evita el modeling_phi3.py cacheado que tiene incompatibilidades con el config actual
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

print("Cargando modelo (float16, device_map=auto)...")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    device_map="auto",
)

print("Creando pipeline de generación de texto...")
pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
)

print("\nModelo cargado correctamente.")
if hasattr(model, 'hf_device_map'):
    print(f"Dispositivos usados: {set(model.hf_device_map.values())}")

Cargando tokenizador de 'microsoft/Phi-3-mini-4k-instruct'...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_token.py:89: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Cargando modelo (float16, device_map=auto)...


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Creando pipeline de generación de texto...

Modelo cargado correctamente.
Dispositivos usados: {0}


### 2.1 Test de conexión

Verificamos que el modelo responde correctamente con una pregunta neutral antes de lanzar el experimento.

In [5]:
test_messages = [{"role": "user", "content": "¿Cuál es la capital de Francia? Responde en una frase."}]

test_output = pipe(test_messages, max_new_tokens=50, do_sample=False)
test_respuesta = test_output[0]["generated_text"][-1]["content"]

print("=== TEST DE CONEXIÓN ===")
print(f"Pregunta: ¿Cuál es la capital de Francia?")
print(f"Respuesta: {test_respuesta}")
print("========================")

You are not running the flash-attention implementation, expect numerical differences.


=== TEST DE CONEXIÓN ===
Pregunta: ¿Cuál es la capital de Francia?
Respuesta:  La capital de Francia es París.


---
## 3. Definición de prompts <a id='sec4'></a>

Los tres prompts abordan el **mismo problema** con diferente nivel de estructura y detalle. Esto nos permite aislar el efecto de la estrategia de prompting en la calidad de la respuesta.

### Mensaje: Reseña
Reseña ssobre la cual los prompts la interpretarán

### Prompt 1 — Base
Instrucción directa y mínima, sin estructura formal ni ejemplos. Sirve como línea base para la comparación.

### Prompt 2 — Plantilla estructurada
Sigue el esquema enseñado en clase: **Tarea + Contexto + Restricciones + Formato de salida + Criterio de calidad**. Proporciona información explícita sobre cómo responder.

### Prompt 3 — Razonamiento paso a paso
Solicita al modelo que analice el texto en etapas, contraste alternativas y justifique su conclusión. Favorece la transparencia del proceso de decisión.

In [6]:
TEXTO = (
    "El producto llegó antes de lo esperado y el embalaje estaba perfecto. "
    "Sin embargo, después de usarlo tres días, dejó de funcionar correctamente. "
    "El servicio de atención al cliente no respondió a mis mensajes. "
    "Muy decepcionante."
)
ETIQUETA_GOLD = "negativa"

print(f"Texto: {TEXTO}")
print(f"Etiqueta gold: {ETIQUETA_GOLD}")

Texto: El producto llegó antes de lo esperado y el embalaje estaba perfecto. Sin embargo, después de usarlo tres días, dejó de funcionar correctamente. El servicio de atención al cliente no respondió a mis mensajes. Muy decepcionante.
Etiqueta gold: negativa


In [7]:
PROMPT_BASE = f"""Analiza el siguiente mensaje y clasifícalo según:
1. Sentimiento: tristeza, alegría o neutralidad
2. Subjetividad: objetivo o subjetivo
3. Polaridad: positivo, negativo o neutro

Mensaje: "{TEXTO}"""

PROMPT_PLANTILLA = f"""Tarea: Clasifica el siguiente mensaje según su sentimiento, subjetividad y polaridad.

Contexto: Eres un sistema de análisis de sentimientos integrado en una plataforma de comunicación empresarial. Los mensajes son textos en español escritos por usuarios reales en distintos contextos.

Restricciones:
- Usa únicamente la información contenida en el mensaje proporcionado.
- No hagas suposiciones sobre el autor ni el contexto externo.
- Responde de forma concisa y estructurada.

Formato de salida:
- Sentimiento: [tristeza / alegría / neutralidad]
- Subjetividad: [objetivo / subjetivo]
- Polaridad: [positivo / negativo / neutro]
- Justificación: [máximo 2 frases explicando la clasificación]

Criterio de calidad: La clasificación debe reflejar el tono predominante del mensaje, no elementos aislados.

Mensaje: "{TEXTO}"""

PROMPT_RAZONAMIENTO = f"""Analiza el siguiente mensaje paso a paso:

Paso 1: Identifica las palabras o expresiones con carga emocional positiva presentes en el mensaje.
Paso 2: Identifica las palabras o expresiones con carga emocional negativa presentes en el mensaje.
Paso 3: Evalúa si el mensaje expresa opiniones o sentimientos personales (subjetivo) o si describe hechos verificables (objetivo).
Paso 4: Considera las tres opciones de sentimiento (tristeza, alegría, neutralidad) y argumenta cuál se ajusta mejor al tono global.
Paso 5: Determina la polaridad (positivo, negativo, neutro) comparando el peso de los elementos identificados.
Paso 6: Proporciona una conclusión final justificada.

Formato de respuesta:
- Sentimiento: [tristeza / alegría / neutralidad]
- Subjetividad: [objetivo / subjetivo]
- Polaridad: [positivo / negativo / neutro]
- Razonamiento: [síntesis breve del proceso de análisis]

Mensaje: "{TEXTO}"""

prompts = {
    "base": PROMPT_BASE,
    "plantilla": PROMPT_PLANTILLA,
    "razonamiento": PROMPT_RAZONAMIENTO,
}

print("Prompts definidos:", list(prompts.keys()))

Prompts definidos: ['base', 'plantilla', 'razonamiento']


### 4.1 Vista previa de los tres prompts aplicados al mensaje 1

In [8]:
for nombre, prompt_text in prompts.items():
    print(f"\n{'='*60}")
    print(f"PROMPT: {nombre.upper()}")
    print('='*60)
    print(prompt_text)
    print()


PROMPT: BASE
Analiza el siguiente mensaje y clasifícalo según:
1. Sentimiento: tristeza, alegría o neutralidad
2. Subjetividad: objetivo o subjetivo
3. Polaridad: positivo, negativo o neutro

Mensaje: "El producto llegó antes de lo esperado y el embalaje estaba perfecto. Sin embargo, después de usarlo tres días, dejó de funcionar correctamente. El servicio de atención al cliente no respondió a mis mensajes. Muy decepcionante.


PROMPT: PLANTILLA
Tarea: Clasifica el siguiente mensaje según su sentimiento, subjetividad y polaridad.

Contexto: Eres un sistema de análisis de sentimientos integrado en una plataforma de comunicación empresarial. Los mensajes son textos en español escritos por usuarios reales en distintos contextos.

Restricciones:
- Usa únicamente la información contenida en el mensaje proporcionado.
- No hagas suposiciones sobre el autor ni el contexto externo.
- Responde de forma concisa y estructurada.

Formato de salida:
- Sentimiento: [tristeza / alegría / neutralidad]

---
## 5. Parámetros y función de inferencia <a id='sec5'></a>

Configuramos los hiperparámetros de generación. La temperatura > 0 es un **requisito obligatorio** de la práctica para poder estudiar la variabilidad entre ejecuciones.

In [9]:
# Verificar que el modelo está en GPU antes de lanzar el experimento
device_modelo = next(model.parameters()).device
print(f"Modelo en: {device_modelo}")
if str(device_modelo) == 'cpu':
    print("ADVERTENCIA: el modelo está en CPU — activa T4 GPU en Colab")
else:
    vram_usada = torch.cuda.memory_allocated() / 1e9
    print(f"VRAM usada por el modelo: {vram_usada:.1f} GB")

GEN_PARAMS = {
    "temperature": 0.7,
    "top_p": 0.9,
    "do_sample": True,
    "max_new_tokens": 200,
}

N_REPETICIONES = 5
BATCH_SIZE = 4  # número de prompts procesados en paralelo en GPU

print(f"\nTotal llamadas: {len(prompts) * N_REPETICIONES}")
print(f"Batch size: {BATCH_SIZE}")

Modelo en: cuda:0
VRAM usada por el modelo: 7.7 GB

Total llamadas: 15
Batch size: 4


---
## 6. Ejecución del experimento <a id='sec6'></a>

Ejecutamos todas las combinaciones: **5 mensajes × 3 prompts × 5 repeticiones = 75 llamadas**.

Los resultados se guardan progresivamente en una lista y se exportan a CSV al finalizar.

In [10]:
# Pre-generar todos los prompts como lista plana
all_inputs = []
all_metadata = []

for prompt_nombre, prompt_text in prompts.items():
    for rep in range(1, N_REPETICIONES + 1):
        all_inputs.append([{"role": "user", "content": prompt_text}])
        all_metadata.append({
            "tipo_prompt": prompt_nombre,
            "repeticion": rep,
            "gold_clasificacion": ETIQUETA_GOLD,
        })

print(f"Prompts preparados: {len(all_inputs)} | Batch size: {BATCH_SIZE}")
print("Ejecutando experimento en GPU...")

# Una sola llamada al pipeline con todos los inputs — elimina el warning de uso secuencial
t_inicio_total = time.time()
outputs = pipe(all_inputs, batch_size=BATCH_SIZE, **GEN_PARAMS)
t_total = time.time() - t_inicio_total

t_medio = t_total / len(all_inputs)

# Reconstruir resultados emparejando outputs con metadata
resultados = []
for output, meta in zip(outputs, all_metadata):
    respuesta = output[0]["generated_text"][-1]["content"]
    resultados.append({
        **meta,
        "respuesta": respuesta,
        "tiempo_s": round(t_medio, 2),
        "longitud_palabras": len(respuesta.split()),
    })

df_resultados = pd.DataFrame(resultados)
df_resultados.to_csv("resultados_phi3_mini.csv", index=False)

print(f"\nExperimento completado en {t_total/60:.1f} minutos.")
print(f"Tiempo medio por llamada: {t_medio:.1f}s")
print(f"Resultados guardados en 'resultados_phi3_mini.csv'")
display(df_resultados[['tipo_prompt', 'repeticion', 'tiempo_s', 'longitud_palabras']])


Prompts preparados: 15 | Batch size: 4
Ejecutando experimento en GPU...

Experimento completado en 0.7 minutos.
Tiempo medio por llamada: 3.0s
Resultados guardados en 'resultados_phi3_mini.csv'


,tipo_prompt,repeticion,tiempo_s,longitud_palabras
0,base,1,2.97,9
1,base,2,2.97,9
2,base,3,2.97,95
3,base,4,2.97,9
4,base,5,2.97,107
5,plantilla,1,2.97,33
6,plantilla,2,2.97,49
7,plantilla,3,2.97,94
8,plantilla,4,2.97,44
9,plantilla,5,2.97,43


### 6.1 Muestra de respuestas por tipo de prompt

Inspeccionamos una respuesta representativa de cada tipo de prompt para el mensaje 1.

In [12]:
print(f"\n{'#'*65}")
print(f"  MENSAJE: {TEXTO[:70]}...")
print(f"  Gold → clasificacion: {ETIQUETA_GOLD}")
print(f"{'#'*65}")
for tipo in ['base', 'plantilla', 'razonamiento']:
    fila = df_resultados[
        (df_resultados['tipo_prompt'] == tipo) &
        (df_resultados['repeticion'] == 1)
    ]
    if fila.empty:
        continue
    print(f"\n  ── PROMPT: {tipo.upper()} ──")
    print(fila['respuesta'].values[0])



#################################################################
  MENSAJE: El producto llegó antes de lo esperado y el embalaje estaba perfecto. ...
  Gold → clasificacion: negativa
#################################################################

  ── PROMPT: BASE ──
 1. Sentimiento: tristeza

2. Subjetividad: subjetivo

3. Polaridad: negativo

  ── PROMPT: PLANTILLA ──
 - Sentimiento: tristeza

- Subjetividad: subjetivo

- Polaridad: negativo

- Justificación: El mensaje expresa decepción y frustración con el producto y el servicio al cliente, lo que indica un tono negativo y subjetivo.

  ── PROMPT: RAZONAMIENTO ──
 - Sentimiento: tristeza

- Subjetividad: subjetivo

- Polaridad: negativo

- Razonamiento: El mensaje contiene palabras con carga emocional positiva como "llegó antes de lo esperado" y "embalaje estaba perfecto", pero se balancea con cargas negativas como "dejó de funcionar correctamente", "no respondió a mis mensajes" y "muy decepcionante". La evaluación de la subje

---
## 7. Métricas objetivas <a id='sec7'></a>

Las métricas objetivas son cuantificables y reproducibles. Medimos:

- **Exactitud**: porcentaje de clasificaciones que coinciden con las etiquetas gold
- **Cumplimiento de formato**: porcentaje de respuestas que incluyen las tres etiquetas esperadas
- **Consistencia**: porcentaje de repeticiones que producen la misma etiqueta para un mismo mensaje y prompt
- **Longitud de respuesta**: media y desviación estándar en número de palabras
- **Tiempo de inferencia**: media en segundos

In [13]:
def normalize(text: str) -> str:
    text = text.lower()
    text = unicodedata.normalize('NFD', text)
    text = ''.join(c for c in text if unicodedata.category(c) != 'Mn')
    return text


def extract_clasificacion(response: str):
    r = normalize(response)
    # Buscar en línea etiquetada: "Clasificación: ..." o "clasificación final: ..."
    m = re.search(
        r'clasificaci[o]n\s*(?:final)?\s*[:\-]\s*\[?(positiv[ao]|negativ[ao]|neutr[ao]|neutral)\]?',
        r
    )
    if m:
        v = m.group(1)
        if v.startswith('positiv'): return 'positiva'
        if v.startswith('negativ'): return 'negativa'
        return 'neutra'
    # Búsqueda en texto libre con variantes de género
    if re.search(r'negativ[ao]', r): return 'negativa'
    if re.search(r'positiv[ao]', r): return 'positiva'
    if re.search(r'neutr[ao]|neutral', r): return 'neutra'
    return None


def check_format_compliance(response: str) -> bool:
    r = normalize(response)
    return bool(re.search(r'clasificaci[o]n|positiv[ao]|negativ[ao]|neutr[ao]|neutral', r))


# Aplicar extracción
df_resultados['pred_clasificacion'] = df_resultados['respuesta'].apply(extract_clasificacion)
df_resultados['formato_ok']         = df_resultados['respuesta'].apply(check_format_compliance)

df_resultados['gold_clasificacion'] = df_resultados['gold_clasificacion'].apply(normalize)

df_resultados['correcto_clasificacion'] = (
    df_resultados['pred_clasificacion'] == df_resultados['gold_clasificacion']
)

# Resumen
no_clas = df_resultados['pred_clasificacion'].isna().sum()
total   = len(df_resultados)
print(f'Etiquetas no extraídas — clasificacion: {no_clas}/{total}')

if no_clas > 0:
    print(f'\n⚠ {no_clas} respuestas sin clasificación extraída. Primeros 2 ejemplos:')
    fallos = df_resultados[df_resultados['pred_clasificacion'].isna()]
    for _, row in fallos.head(2).iterrows():
        print(f'\n  msg={row["mensaje_id"]} | prompt={row["tipo_prompt"]} | rep={row["repeticion"]}')
        print(f'  Respuesta: {row["respuesta"][:300]}')
else:
    print('Todas las etiquetas de clasificación extraídas correctamente.')


Etiquetas no extraídas — clasificacion: 0/15
Todas las etiquetas de clasificación extraídas correctamente.


In [14]:
# Tabla resumen de métricas objetivas por tipo de prompt
resumen_objetivo = df_resultados.groupby("tipo_prompt").agg(
    exactitud_clasificacion=("correcto_clasificacion", "mean"),
    cumplimiento_formato=("formato_ok", "mean"),
    longitud_media_palabras=("longitud_palabras", "mean"),
    longitud_std_palabras=("longitud_palabras", "std"),
    tiempo_medio_s=("tiempo_s", "mean"),
).round(3)

resumen_objetivo = resumen_objetivo.reindex(["base", "plantilla", "razonamiento"])

print("=== RESUMEN MÉTRICAS OBJETIVAS POR TIPO DE PROMPT ===")
display(resumen_objetivo)

=== RESUMEN MÉTRICAS OBJETIVAS POR TIPO DE PROMPT ===


,exactitud_clasificacion,cumplimiento_formato,longitud_media_palabras,longitud_std_palabras,tiempo_medio_s
tipo_prompt,,,,,
base,1.0,1.0,45.8,50.569,2.97
plantilla,1.0,1.0,52.6,23.860,2.97
razonamiento,1.0,1.0,93.2,14.856,2.97


In [17]:
def calc_consistency(series):
    if series.isna().all():
        return np.nan
    return series.value_counts().iloc[0] / len(series)


consistencia_resumen = df_resultados.groupby("tipo_prompt").agg(
    consist_clasificacion=("pred_clasificacion", calc_consistency),
).round(3)
consistencia_resumen = consistencia_resumen.reindex(["base", "plantilla", "razonamiento"])

print("=== CONSISTENCIA MEDIA POR TIPO DE PROMPT ===")
print("(1.0 = misma etiqueta en las 5 repeticiones, 0.2 = cada repetición diferente)")
display(consistencia_resumen)

=== CONSISTENCIA MEDIA POR TIPO DE PROMPT ===
(1.0 = misma etiqueta en las 5 repeticiones, 0.2 = cada repetición diferente)


,consist_clasificacion
tipo_prompt,
base,1.0
plantilla,1.0
razonamiento,1.0


---
## 8. Análisis de variabilidad <a id='sec9'></a>

Estudiamos cómo varía el comportamiento del modelo entre repeticiones del **mismo prompt sobre el mismo mensaje**.

Preguntas que responde este análisis:
- ¿Mantiene el modelo el mismo criterio de clasificación entre repeticiones?
- ¿El prompt estructurado reduce la variabilidad?
- ¿Hay mensajes más "difíciles" (más inconsistentes) que otros?

In [18]:
print("=== DISTRIBUCIÓN DE ETIQUETAS PREDICHAS POR TIPO DE PROMPT ===")
tabla_clas = pd.crosstab(
    df_resultados["tipo_prompt"],
    df_resultados["pred_clasificacion"],
    margins=False
)
display(tabla_clas)

=== DISTRIBUCIÓN DE ETIQUETAS PREDICHAS POR TIPO DE PROMPT ===


pred_clasificacion,negativa
tipo_prompt,
base,5
plantilla,5
razonamiento,5


In [19]:
# Variabilidad en longitud de respuesta
print("=== VARIABILIDAD EN LONGITUD DE RESPUESTA (palabras) ===")
variabilidad_longitud = df_resultados.groupby("tipo_prompt")["longitud_palabras"].agg(
    media="mean",
    std="std",
    minimo="min",
    maximo="max"
).round(1)
variabilidad_longitud = variabilidad_longitud.reindex(["base", "plantilla", "razonamiento"])
display(variabilidad_longitud)

print("\nInterpretación: una desviación estándar alta indica que el modelo no produce")
print("respuestas de longitud consistente con este tipo de prompt.")

=== VARIABILIDAD EN LONGITUD DE RESPUESTA (palabras) ===


,media,std,minimo,maximo
tipo_prompt,,,,
base,45.8,50.6,9,107
plantilla,52.6,23.9,33,94
razonamiento,93.2,14.9,69,108



Interpretación: una desviación estándar alta indica que el modelo no produce
respuestas de longitud consistente con este tipo de prompt.


---
## 9. Conclusiones <a id='sec11'></a>

### 9.1 Resultados principales

> *(Completar tras revisar los resultados del experimento)*

**Exactitud global:**
- Prompt base: ...
- Prompt plantilla: ...
- Prompt razonamiento: ...

**Consistencia:**
- El modelo muestra mayor/menor estabilidad con el prompt ...
- Los mensajes más inconsistentes fueron ...

### 9.2 Comparación entre estrategias de prompting

> *(Completar)*

- El **prompt base** produce respuestas ... porque ...
- El **prompt plantilla** mejora/empeora ... porque ...
- El **prompt de razonamiento** destaca en ... aunque introduce ...

### 9.3 Comportamiento del modelo Phi-3-mini

> *(Completar)*

- Fortalezas observadas: ...
- Debilidades observadas: ...
- Casos en los que el modelo falla sistemáticamente: ...

### 9.4 Limitaciones del experimento

- Dataset reducido: los resultados no son estadísticamente representativos.
- La evaluación subjetiva depende del criterio del evaluador.
- Los resultados pueden variar entre ejecuciones por la naturaleza estocástica del modelo (temperature=0.7).
- El modelo puede tener sesgos derivados de su entrenamiento que afectan a categorías específicas.

### 10.5 Reflexión final

> *(Completar con las conclusiones del grupo)*